# 综合实训：智能客服工单分发系统

## 1. 学习目标

完成本课程后，你应该能够：

- 独立完成一个文本分类任务（意图识别）
- 使用**正则表达式**提取关键信息（如订单号）
- 将 NLP 模型封装成一个简单的应用功能
- 解决实际业务场景中的效率问题

## 2. 项目介绍

### 2.1 项目背景

**场景设定：**  
你是一家电商公司的技术专员。双十一大促期间，客服部门崩溃了！每天有成千上万条客户咨询涌入，客服小姐姐们需要人工手动做两件事：
1.  **分单**：看客户是问物流、售后还是售前，然后转给对应的小组。
2.  **录入**：手动复制客户聊天记录里的“订单号”，粘贴到系统里去查单。

**老板的需求：**
“给我做一个**AI 客服主管**，自动把工单分发给对应部门，并且把订单号自动提取出来！”

### 2.2 数据集介绍

**数据来源：** 真实客服对话脱敏数据  
**数据规模：** 150条精选记录     
**数据文件：** `/home/jovyan/work/datasets/691fd2ed6639511b893dc462-momodel/customer_service_data.csv`     

**数据字段说明：**

| 列名 | 说明 | 示例 |
|------|------|------|
| **label** | **咨询意图（标签）** | 物流咨询、售后退换、售前咨询 |
| **content** | **用户咨询内容** | "我的快递怎么还没到？单号SF123456" |


## 3. 步骤1：加载和探索数据

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# 1. 读取数据
df = pd.read_csv('/home/jovyan/work/datasets/691fd2ed6639511b893dc462-momodel/customer_service_data.csv')

print("数据加载完成！")
print(f"\n数据集基本信息：")
print(f"总记录数: {len(df)}")
print(f"\n前5条数据预览：")
print(df.head())

### 3.1 意图分布分析
我们需要看看，客户主要都在问哪类问题？

In [ ]:
# 统计各类咨询的数量
label_counts = df['label'].value_counts()

print("咨询意图分布：")
print("="*40)
for label, count in label_counts.items():
    percentage = count / len(df) * 100
    bar = "█" * int(percentage / 5)
    print(f"{label:<6} {count:<5} {percentage:.1f}%  {bar}")

## 4. 步骤2：挑战一——“大海捞针”（实体提取）

**痛点：** 客服最烦的就是在一大段话里找单号。  
**解决方案：** 使用**正则表达式 (Regular Expression)**。这是 NLP 中最基础但也最实用的技能。

**观察单号规律：**
我们发现单号通常长这样：`SF123456`、`TB999888`、`JD111222`。
规律是：**2个大写字母 + 一串数字**。

In [ ]:
import re # 导入正则库

def extract_order_id(text):
    """
    功能：从文本中提取订单号
    规则：2个大写字母 + 连续的数字
    """
    # [A-Z]{2} 表示匹配2个大写字母
    # \d+      表示匹配1个或多个数字
    pattern = r'[A-Z]{2}\d+'
    
    # re.findall 会找到所有符合规则的字符串
    ids = re.findall(pattern, text)
    
    if ids:
        return ids[0] # 返回找到的第一个
    else:
        return None

# --- 测试一下 ---
test_msg_1 = "我的单号是SF102030，麻烦查一下"
test_msg_2 = "这件衣服有现货吗？"

print(f"测试文本1：{test_msg_1}")
print(f"提取结果：{extract_order_id(test_msg_1)}")
print("-"*30)
print(f"测试文本2：{test_msg_2}")
print(f"提取结果：{extract_order_id(test_msg_2)}")

## 5. 步骤3：挑战二——“读懂人心”（意图分类）

**任务：** 训练一个模型，自动判断客户想干什么（物流？售后？售前？）。

我们将使用经典的机器学习流程：
1.  **分词**：把句子切开。
2.  **特征提取 (TF-IDF)**：把文字变成向量。
3.  **模型训练 (朴素贝叶斯)**：让计算机学会分类。

In [ ]:
import jieba
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

# 1. 数据预处理：中文分词
print("正在进行分词处理...")
# 列表推导式：对每一行content进行分词，并用空格连接
df['clean_content'] = [' '.join(jieba.lcut(text)) for text in df['content']]

print(f"分词示例：{df['clean_content'][0]}")

# 2. 划分训练集和测试集
X_train, X_test, y_train, y_test = train_test_split(
    df['clean_content'], 
    df['label'], 
    test_size=0.2, 
    random_state=42
)

# 3. 特征提取 (TF-IDF)
vectorizer = TfidfVectorizer()
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

# 4. 训练模型
print("\n正在训练朴素贝叶斯模型...")
model = MultinomialNB()
model.fit(X_train_vec, y_train)

print("✅ 模型训练完成！")

## 6. 步骤4：模型评估
我们需要看看这个“AI客服主管”的水平如何，能不能上岗？

In [ ]:
# 预测测试集
y_pred = model.predict(X_test_vec)

# 计算准确率
acc = accuracy_score(y_test, y_pred)
print(f"模型准确率：{acc:.2%}")

# 打印详细报告
print("\n详细分类报告：")
print(classification_report(y_test, y_pred))

## 7. 步骤5：综合应用——打造“AI自动分单系统”

最后，我们将**正则提取**和**分类模型**结合起来，写一个完整的函数。
这个函数就是我们最终交付给老板的“产品”。

In [ ]:
def ai_supervisor(user_input):
    """
    AI 客服主管的主逻辑函数
    """
    print("--------------------------------------------------")
    print(f"📞 接到用户咨询：{user_input}")
    
    # 1. 提取单号 (规则派)
    order_id = extract_order_id(user_input)
    
    # 2. 判断意图 (模型派)
    # 记得要先分词，再转向量，步骤不能少
    input_seg = " ".join(jieba.lcut(user_input))
    input_vec = vectorizer.transform([input_seg])
    prediction = model.predict(input_vec)[0]
    
    # 3. 自动分单操作
    print(f"🤖 意图识别结果：【{prediction}】")
    
    if order_id:
        print(f"📋 关键信息抓取：检测到单号 [{order_id}]")
        print(f"✅ 自动处理：已将单号 {order_id} 录入系统，并转接至 [{prediction}组]。")
    else:
        print(f"⚠️ 信息缺失：未检测到单号。")
        print(f"✅ 自动处理：已将对话转接至 [{prediction}组] 人工客服。")

print("系统已启动，随时准备工作！")

### 7.1 模拟真实场景测试
让我们模拟几条用户发来的消息，看看 AI 的反应。

In [ ]:
# 模拟用户消息队列
messages = [
    "我的快递怎么还没到？单号SF102030，急死我了！",
    "这衣服有色差，我要退货，单号TB999888。",
    "请问这款鞋子偏码吗？平时穿37的。",
    "查一下物流，JD123123，一直不动。",
    "你是机器人吗？我要找人工！"
]

for msg in messages:
    ai_supervisor(msg)
    print("\n")

## 8. 项目总结

### 8.1 我们完成了什么？

1.  **数据处理**：处理了150条模拟的真实客服语料。
2.  **关键技能一**：学会了用**正则表达式**提取固定格式的信息（订单号），这在工作中超级实用。
3.  **关键技能二**：学会了用**朴素贝叶斯**训练文本分类模型，准确率令人满意。
4.  **系统集成**：不仅是跑模型，而是将技术封装成了解决实际问题的“AI主管”程序。

### 8.2 课后思考

- 如果用户发的是“我的手机号是13800138000”，现在的正则能提取吗？应该怎么改？
- 如果出现新的咨询类型（比如“投诉建议”），模型该如何更新？